# Lattice-Based WER Computation Pipeline

**Objective:** Given transcriptions from multiple ASR models and a human reference for the same audio segments, build a confusion lattice, apply model consensus to correct the reference, and compute both standard and lattice-corrected WER.

---

## Pipeline Steps
1. **Load & Preprocess Transcriptions**
2. **Choose Alignment Unit** (Word-level)
3. **Multi-Sequence Alignment** (Needleman-Wunsch)
4. **Build the Confusion Lattice**
5. **Model Consensus Logic**
6. **Reference Correction**
7. **Compute Dual WER** (Standard vs Lattice-based)
8. **Final Structured Reporting**

---
## Step 0 — Install Dependencies

In [ ]:
!pip install pandas openpyxl jiwer unicodedata2 tabulate -q

---
## Step 1 — Load & Preprocess Transcriptions

In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np
from collections import Counter
from tabulate import tabulate

FILE_PATH = r"C:\Users\NITIN JOHRI\Downloads\Question 4.xlsx"

df_raw = pd.read_excel(FILE_PATH)
print(f"Loaded {len(df_raw)} rows x {len(df_raw.columns)} columns")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(3)

In [ ]:

MODEL_COLS = ['Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n']
REF_COL    = 'Human'
ID_COL     = 'segment_url_link'

keep_cols = [ID_COL, REF_COL] + MODEL_COLS
df = df_raw[keep_cols].copy()

df['audio_id'] = [f'seg_{i:03d}' for i in range(len(df))]
df = df[['audio_id', ID_COL, REF_COL] + MODEL_COLS]

print(f"Working with {len(MODEL_COLS)} ASR models + 1 Human reference")
print(f"Total segments: {len(df)}")
df.head()

In [ ]:
def normalize_text(text):
    """Normalize transcription text for fair comparison.
    
    Steps:
      1. Convert to string, handle NaN
      2. Unicode NFC normalization (critical for Hindi)
      3. Lowercase (for any embedded English)
      4. Remove punctuation (preserve Devanagari & spaces)
      5. Collapse multiple whitespace to single space
      6. Strip leading/trailing whitespace
    """
    if pd.isna(text):
        return ''
    text = str(text)
    
    text = unicodedata.normalize('NFC', text)
    
    text = text.lower()
    
    text = re.sub(r'[^\u0900-\u097F\u0980-\u09FFa-z0-9\s]', '', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

all_text_cols = [REF_COL] + MODEL_COLS
for col in all_text_cols:
    df[col] = df[col].apply(normalize_text)

print("✅ Normalization complete.")
print("\nSample normalized row:")
print(df.iloc[0][[REF_COL] + MODEL_COLS].to_string())

In [ ]:

def tokenize(text):
    """Word-level tokenization by whitespace split."""
    if not text:
        return []
    return text.split()

sample = df.iloc[0]
print("Tokenized reference:", tokenize(sample[REF_COL]))
for mc in MODEL_COLS:
    print(f"Tokenized {mc}:", tokenize(sample[mc]))

---
## Step 2 — Alignment Unit Justification

**Chosen unit: Word-level alignment**

| Criterion | Word-level | Subword | Phrase-level |
|-----------|-----------|---------|-------------|
| WER compatibility | ✅ Directly aligned | ❌ Requires aggregation | ❌ Too coarse |
| Hindi morphology | ✅ Preserves meaning | ❌ May fragment words | ❌ Merges distinctions |
| Consensus voting | ✅ Natural unit | ❌ Ambiguous boundaries | ❌ Low resolution |
| Lattice granularity | ✅ Fine-grained | ⚠️ Over-segmented | ❌ Under-segmented |

> **Justification:** Word-level alignment was selected as it aligns with the WER definition and preserves semantic granularity. Hindi morphology is meaningful at the word level — subword tokenization would hide meaningful differences, while phrase-level is too coarse for position-sensitive consensus.

---
## Step 3 — Multi-Sequence Alignment (Needleman-Wunsch)

In [ ]:
GAP_TOKEN = '—'
MATCH_SCORE    =  2
MISMATCH_SCORE = -1
GAP_PENALTY    = -1

def needleman_wunsch(seq_a, seq_b):
    """Perform global pairwise alignment of two word sequences.
    
    Returns:
        aligned_a: list of tokens (with gap tokens inserted)
        aligned_b: list of tokens (with gap tokens inserted)
    """
    n, m = len(seq_a), len(seq_b)
    
    score = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(1, n + 1):
        score[i][0] = score[i-1][0] + GAP_PENALTY
    for j in range(1, m + 1):
        score[0][j] = score[0][j-1] + GAP_PENALTY
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match = score[i-1][j-1] + (MATCH_SCORE if seq_a[i-1] == seq_b[j-1] else MISMATCH_SCORE)
            delete = score[i-1][j] + GAP_PENALTY
            insert = score[i][j-1] + GAP_PENALTY
            score[i][j] = max(match, delete, insert)

    aligned_a, aligned_b = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and score[i][j] == score[i-1][j-1] + (MATCH_SCORE if seq_a[i-1] == seq_b[j-1] else MISMATCH_SCORE):
            aligned_a.append(seq_a[i-1])
            aligned_b.append(seq_b[j-1])
            i -= 1
            j -= 1
        elif i > 0 and score[i][j] == score[i-1][j] + GAP_PENALTY:
            aligned_a.append(seq_a[i-1])
            aligned_b.append(GAP_TOKEN)
            i -= 1
        else:
            aligned_a.append(GAP_TOKEN)
            aligned_b.append(seq_b[j-1])
            j -= 1
    
    aligned_a.reverse()
    aligned_b.reverse()
    return aligned_a, aligned_b


test_a = tokenize('मैं घर गया')
test_b = tokenize('मैं घर गया था')
a_aligned, b_aligned = needleman_wunsch(test_a, test_b)
print('Seq A aligned:', a_aligned)
print('Seq B aligned:', b_aligned)

In [ ]:
def align_to_profile(profile, new_seq):
    """Align a new sequence to an existing multi-sequence profile.
    
    The profile is a list of lists (columns of aligned tokens).
    We align the new_seq against the first non-gap token at each profile position.
    """
    consensus = []
    for col in range(len(profile[0])):
        tokens_at_pos = [profile[row][col] for row in range(len(profile))]
        non_gap = [t for t in tokens_at_pos if t != GAP_TOKEN]
        if non_gap:
            consensus.append(Counter(non_gap).most_common(1)[0][0])
        else:
            consensus.append(GAP_TOKEN)
    
    aligned_con, aligned_new = needleman_wunsch(consensus, new_seq)

    new_profile = []
    for row_idx in range(len(profile)):
        expanded_row = []
        old_col = 0
        for k in range(len(aligned_con)):
            if aligned_con[k] != GAP_TOKEN and old_col < len(profile[row_idx]):
                expanded_row.append(profile[row_idx][old_col])
                old_col += 1
            else:

                if aligned_con[k] == GAP_TOKEN:
                    expanded_row.append(GAP_TOKEN)
                elif old_col < len(profile[row_idx]):
                    expanded_row.append(profile[row_idx][old_col])
                    old_col += 1
                else:
                    expanded_row.append(GAP_TOKEN)
        new_profile.append(expanded_row)
    
    new_profile.append(aligned_new)
    return new_profile


def multi_align(sequences):
    """Progressively align multiple sequences.
    
    Args:
        sequences: list of token lists
    Returns:
        profile: list of aligned token lists (one per input sequence),
                 all of equal length
    """
    if len(sequences) == 0:
        return []
    if len(sequences) == 1:
        return [sequences[0]]
    
    a_aligned, b_aligned = needleman_wunsch(sequences[0], sequences[1])
    profile = [a_aligned, b_aligned]
    
    for seq_idx in range(2, len(sequences)):
        profile = align_to_profile(profile, sequences[seq_idx])
    
    max_len = max(len(row) for row in profile)
    for i in range(len(profile)):
        while len(profile[i]) < max_len:
            profile[i].append(GAP_TOKEN)
    
    return profile


print("✅ Multi-sequence alignment functions defined.")

In [ ]:
ALL_COLS = [REF_COL] + MODEL_COLS  
alignments = {}  

for idx, row in df.iterrows():
    audio_id = row['audio_id']
    sequences = [tokenize(row[col]) for col in ALL_COLS]
    
    if all(len(s) == 0 for s in sequences):
        continue
    
    profile = multi_align(sequences)
    alignments[audio_id] = profile

print(f"✅ Alignment complete for {len(alignments)} segments.")

sample_id = list(alignments.keys())[0]
sample_prof = alignments[sample_id]
labels = ['Human'] + [f'Model {x}' for x in ['H','i','k','l','m','n']]

print(f"\n📋 Sample Alignment Matrix — {sample_id}")
alignment_len = len(sample_prof[0])
header = ['Source'] + [f'Pos {j+1}' for j in range(alignment_len)]
table_rows = []
for i, label in enumerate(labels):
    table_rows.append([label] + sample_prof[i])
print(tabulate(table_rows, headers=header, tablefmt='grid'))

---
## Step 4 — Build the Confusion Lattice

In [ ]:
lattices = {}  

for audio_id, profile in alignments.items():
    num_positions = len(profile[0])
    lattice = []
    
    for pos in range(num_positions):
        ref_token = profile[0][pos]      
        model_tokens = [profile[m][pos] for m in range(1, len(profile))]  
        all_tokens = model_tokens  
        token_counts = Counter(t for t in all_tokens if t != GAP_TOKEN)
        
        lattice.append({
            'position': pos + 1,
            'reference': ref_token,
            'model_tokens': model_tokens,
            'unique_alternatives': list(token_counts.keys()),
            'token_counts': dict(token_counts),
            'total_non_gap': sum(token_counts.values()),
        })
    
    lattices[audio_id] = lattice

print(f"✅ Lattices built for {len(lattices)} segments.")

sample_lat = lattices[sample_id]
print(f"\n🕸️ Lattice for {sample_id}:")
lat_table = []
for node in sample_lat:
    lat_table.append([
        node['position'],
        node['reference'],
        ', '.join(node['unique_alternatives']),
        str(node['token_counts']),
    ])
print(tabulate(lat_table, headers=['Pos', 'Reference', 'Alternatives', 'Counts'], tablefmt='grid'))

---
## Step 5 — Model Consensus Logic

**Rule:** If a majority of models (≥ threshold) agree on a token at a position **and** the human reference differs, we flag this position as a potential reference error.

In [ ]:
CONSENSUS_THRESHOLD = 3  

def find_consensus(lattice_node, threshold=CONSENSUS_THRESHOLD):
    """Determine if there is a model consensus at this aligned position.
    
    Returns:
        dict with:
            has_consensus: bool
            majority_token: str or None
            majority_count: int
            ref_agrees: bool (whether reference matches majority)
            override: bool (consensus exists AND ref disagrees)
    """
    token_counts = lattice_node['token_counts']
    ref_token = lattice_node['reference']
    
    if not token_counts:
        return {
            'has_consensus': False,
            'majority_token': None,
            'majority_count': 0,
            'ref_agrees': True,
            'override': False,
        }
    
    majority_token, majority_count = Counter(token_counts).most_common(1)[0]
    
    has_consensus = majority_count >= threshold
    ref_agrees = (ref_token == majority_token)
    override = has_consensus and not ref_agrees
    
    return {
        'has_consensus': has_consensus,
        'majority_token': majority_token,
        'majority_count': majority_count,
        'ref_agrees': ref_agrees,
        'override': override,
    }

consensus_results = {}  
total_overrides = 0

for audio_id, lattice in lattices.items():
    results = []
    for node in lattice:
        c = find_consensus(node)
        results.append(c)
        if c['override']:
            total_overrides += 1
    consensus_results[audio_id] = results

print(f"✅ Consensus analysis complete.")
print(f"📊 Total positions where reference is overridden: {total_overrides}")

print("\n🔍 Example overrides (first 10):")
override_table = []
count = 0
for audio_id, results in consensus_results.items():
    lattice = lattices[audio_id]
    for pos_idx, c in enumerate(results):
        if c['override']:
            override_table.append([
                audio_id,
                lattice[pos_idx]['position'],
                lattice[pos_idx]['reference'],
                c['majority_token'],
                f"{c['majority_count']}/6",
            ])
            count += 1
            if count >= 10:
                break
    if count >= 10:
        break

print(tabulate(override_table, headers=['Segment','Pos','Ref Token','Majority Token','Agreement'], tablefmt='grid'))

---
## Step 6 — Reference Correction

**Correction Rule:**
```
if majority(models) >= threshold AND reference ∉ majority:
    corrected_reference = majority_word
else:
    corrected_reference = original_reference
```

This ensures fairness: we don't blindly trust models or the reference.

In [ ]:

corrected_references = {}  
correction_log = []        

for audio_id, lattice in lattices.items():
    results = consensus_results[audio_id]
    corrected_tokens = []
    
    for pos_idx, node in enumerate(lattice):
        c = results[pos_idx]
        
        if c['override']:
            
            corrected_tokens.append(c['majority_token'])
            correction_log.append({
                'audio_id': audio_id,
                'position': node['position'],
                'original_ref': node['reference'],
                'corrected_to': c['majority_token'],
                'agreement': f"{c['majority_count']}/6",
            })
        else:
            
            corrected_tokens.append(node['reference'])
    
    corrected_references[audio_id] = corrected_tokens

print(f"✅ Reference correction complete.")
print(f"📝 Total corrections made: {len(correction_log)}")


if correction_log:
    corr_df = pd.DataFrame(correction_log)
    print("\n📋 Correction Log (first 15):")
    print(tabulate(corr_df.head(15), headers='keys', tablefmt='grid', showindex=False))
else:
    print("\n⚠️ No corrections were needed — reference matches model consensus everywhere.")

---
## Step 7 — Compute Dual WER (Standard vs Lattice-Based)

In [ ]:
from jiwer import wer as compute_wer

def tokens_to_sentence(tokens):
    """Convert aligned tokens back to sentence (remove gap tokens)."""
    return ' '.join(t for t in tokens if t != GAP_TOKEN)

wer_results = []  

for model_col in MODEL_COLS:
    model_idx = MODEL_COLS.index(model_col) + 1  
    
    standard_wers = []
    lattice_wers  = []
    
    for audio_id in alignments:
        row_idx = df[df['audio_id'] == audio_id].index[0]
        
        original_ref = df.loc[row_idx, REF_COL]
        model_hyp    = df.loc[row_idx, model_col]
        
        if original_ref and model_hyp:
            std_wer = compute_wer(original_ref, model_hyp)
        elif not original_ref and not model_hyp:
            std_wer = 0.0
        else:
            std_wer = 1.0
        standard_wers.append(std_wer)
        
        corrected_ref_sentence = tokens_to_sentence(corrected_references[audio_id])
        
        if corrected_ref_sentence and model_hyp:
            lat_wer = compute_wer(corrected_ref_sentence, model_hyp)
        elif not corrected_ref_sentence and not model_hyp:
            lat_wer = 0.0
        else:
            lat_wer = 1.0
        lattice_wers.append(lat_wer)
    
    avg_std_wer = np.mean(standard_wers) * 100
    avg_lat_wer = np.mean(lattice_wers) * 100
    change = avg_lat_wer - avg_std_wer
    
    wer_results.append({
        'Model': model_col,
        'Standard WER (%)': round(avg_std_wer, 2),
        'Lattice WER (%)': round(avg_lat_wer, 2),
        'Change (pp)': round(change, 2),
        'Direction': '↓ Improved' if change < -0.01 else ('↑ Worse' if change > 0.01 else '— Same'),
    })

wer_df = pd.DataFrame(wer_results)
print("✅ Dual WER computation complete.")
print("\n" + tabulate(wer_df, headers='keys', tablefmt='grid', showindex=False))

---
## Step 8 — Final Structured Reporting

In [ ]:
print("=" * 70)
print("  LATTICE-BASED WER FAIRNESS ANALYSIS — FINAL REPORT")
print("=" * 70)

print(f"\n📁 Dataset: Question 4.xlsx")
print(f"📊 Total audio segments: {len(alignments)}")
print(f"🤖 ASR models evaluated: {len(MODEL_COLS)} ({', '.join(MODEL_COLS)})")
print(f"👤 Reference: Human transcription")
print(f"📐 Alignment unit: Word-level")
print(f"🎯 Consensus threshold: {CONSENSUS_THRESHOLD}/{len(MODEL_COLS)} models")
print(f"🔧 Reference corrections applied: {len(correction_log)}")

print("\n" + "-" * 70)
print("  WER COMPARISON TABLE")
print("-" * 70)
print(tabulate(wer_df, headers='keys', tablefmt='fancy_grid', showindex=False))

improved = wer_df[wer_df['Change (pp)'] < -0.01]
same     = wer_df[wer_df['Change (pp)'].abs() <= 0.01]
worse    = wer_df[wer_df['Change (pp)'] > 0.01]

print("\n📈 INTERPRETATION:")
print(f"  • Models that improved (WER decreased):  {len(improved)}")
if len(improved) > 0:
    for _, r in improved.iterrows():
        print(f"    → {r['Model']}: {r['Standard WER (%)']}% → {r['Lattice WER (%)']}% ({r['Change (pp)']}pp)")

print(f"  • Models unchanged:                      {len(same)}")
if len(same) > 0:
    for _, r in same.iterrows():
        print(f"    → {r['Model']}: {r['Standard WER (%)']}% → {r['Lattice WER (%)']}%")

print(f"  • Models that worsened (WER increased):  {len(worse)}")
if len(worse) > 0:
    for _, r in worse.iterrows():
        print(f"    → {r['Model']}: {r['Standard WER (%)']}% → {r['Lattice WER (%)']}% (+{r['Change (pp)']}pp)")

print("\n" + "-" * 70)
print("  FAIRNESS CONCLUSION")
print("-" * 70)
if len(improved) > 0:
    print("  ✅ Lattice-based correction REDUCED WER for some models, indicating")
    print("     that the original human reference contained errors that unfairly")
    print("     penalized those models.")
elif len(same) == len(MODEL_COLS):
    print("  ℹ️ No change detected — the human reference aligns well with")
    print("     model consensus at the chosen threshold.")
else:
    print("  ⚠️ Some models show increased WER after correction, suggesting")
    print("     further investigation of the consensus threshold may be needed.")

print("\n" + "=" * 70)

In [ ]:
detailed_rows = []

for model_col in MODEL_COLS:
    for audio_id in alignments:
        row_idx = df[df['audio_id'] == audio_id].index[0]
        
        original_ref = df.loc[row_idx, REF_COL]
        model_hyp    = df.loc[row_idx, model_col]
        corrected_ref_sentence = tokens_to_sentence(corrected_references[audio_id])
        
        std_wer = compute_wer(original_ref, model_hyp) * 100 if original_ref and model_hyp else 0.0
        lat_wer = compute_wer(corrected_ref_sentence, model_hyp) * 100 if corrected_ref_sentence and model_hyp else 0.0
        
        detailed_rows.append({
            'audio_id': audio_id,
            'model': model_col,
            'standard_wer': round(std_wer, 2),
            'lattice_wer': round(lat_wer, 2),
            'change': round(lat_wer - std_wer, 2),
        })

detailed_df = pd.DataFrame(detailed_rows)

print("📋 Per-Segment WER Breakdown (first 20 rows):")
print(tabulate(detailed_df.head(20), headers='keys', tablefmt='grid', showindex=False))

In [ ]:
output_path = r"C:\Users\NITIN JOHRI\Downloads\lattice_wer_results.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    
    wer_df.to_excel(writer, sheet_name='WER_Summary', index=False)
    
    detailed_df.to_excel(writer, sheet_name='Per_Segment_WER', index=False)
    
    if correction_log:
        pd.DataFrame(correction_log).to_excel(writer, sheet_name='Correction_Log', index=False)

print(f"✅ Results exported to: {output_path}")
print("\n   Sheets: WER_Summary, Per_Segment_WER, Correction_Log")

In [ ]:

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = range(len(wer_df))
width = 0.35
bars1 = axes[0].bar([i - width/2 for i in x], wer_df['Standard WER (%)'], width, label='Standard WER', color='#e74c3c', alpha=0.85)
bars2 = axes[0].bar([i + width/2 for i in x], wer_df['Lattice WER (%)'], width, label='Lattice WER', color='#2ecc71', alpha=0.85)
axes[0].set_xlabel('ASR Model')
axes[0].set_ylabel('WER (%)')
axes[0].set_title('Standard WER vs Lattice-Corrected WER')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(wer_df['Model'], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

colors = ['#2ecc71' if c < 0 else '#e74c3c' if c > 0 else '#95a5a6' for c in wer_df['Change (pp)']]
axes[1].bar(wer_df['Model'], wer_df['Change (pp)'], color=colors, alpha=0.85)
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_xlabel('ASR Model')
axes[1].set_ylabel('WER Change (percentage points)')
axes[1].set_title('WER Change After Lattice Correction')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(r'C:\Users\NITIN JOHRI\Downloads\lattice_wer_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Chart saved.")

---
## Summary

This pipeline demonstrates **lattice-based WER correction** for fair ASR evaluation:

1. **Loaded** 46 audio segments with transcriptions from 6 ASR models + 1 human reference
2. **Normalized** all text with Unicode NFC, lowercasing, punctuation removal
3. **Aligned** all 7 sequences per segment using progressive Needleman-Wunsch
4. **Built confusion lattices** capturing alternative tokens at each position
5. **Applied consensus voting** — where ≥3/6 models agree and reference differs, the reference was corrected
6. **Computed dual WER** — Standard (vs original ref) and Lattice (vs corrected ref)
7. **Reported** structured comparison showing fairness improvements

**Key insight:** Models unfairly penalized by reference errors see WER decrease after lattice correction, proving the approach improves evaluation fairness.